# Phase 5: Legal Statutes Prediction

 _This is submission 2 where we do full fine-tuning of Selective SSM model for downstream task of Legal Statutes Prediction._

## 1. Login and GPU check

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Dataloading and Pre-processing

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer


HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone" 

# --- 1. Data Loading ---
print("Loading datasets from Hugging Face Hub...")

try:
    # Explicitly specify 'phase_4' here
    hf_dataset = load_dataset(HF_DATASET_REPO, 'phase_5')
    print(f"Successfully loaded splits: {list(hf_dataset.keys())}")
except Exception as e:
    print(f"Error loading dataset: {e}")

def standardize_labels(example, is_test=False):
    if not is_test:
        if 'pred' in example and 'label' not in example:
             example['label'] = example['pred']
    return example

# Apply standard formatting
for split in hf_dataset.keys():
    is_test_split = (split == 'test')
    hf_dataset[split] = hf_dataset[split].map(lambda x: standardize_labels(x, is_test=is_test_split))

# Assuming you have N total distinct statutes in Phase 5
NUM_STATUTES = 100 

def multi_hot_encode(example):
    # Convert list of statute IDs to a binary vector of size NUM_STATUTES
    labels = np.zeros(NUM_STATUTES, dtype=np.float32)
    if "labels" in example and example["labels"] is not None:
        for statute_id in example["labels"]:
            labels[statute_id] = 1.0
        example["labels"] = labels
    return example

hf_dataset = hf_dataset.map(multi_hot_encode)
print("Data loading complete.")

In [ ]:
# --- 2. Preprocessing & Strategic Truncation ---

from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

BPE_FILE = "/kaggle/input/datasets/vaibhavim081203/phase3feed/cs781_phase1/legal_bpe.json"

# Load backend tokenizer directly
backend_tok = Tokenizer.from_file(BPE_FILE)

# Wrap into Hugging Face PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tok)

# Define special tokens according to your Phase 1 setup
tokenizer.pad_token = "[PAD]" if "[PAD]" in tokenizer.get_vocab() else tokenizer.eos_token
tokenizer.cls_token = "[CLS]" if "[CLS]" in tokenizer.get_vocab() else "<s>"
tokenizer.sep_token = "[SEP]" if "[SEP]" in tokenizer.get_vocab() else "</s>"

print(f"Tokenizer loaded successfully! Vocab size: {len(tokenizer)}")

# Safe fallback for special tokens
START_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0)
END_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

MAX_LENGTH = 4096 # The SSM can easily handle 4k+ tokens in linear time
NUM_STATUTES = 100 # Update this with your exact number of unique statutes

# Assuming hf_dataset is already loaded and multi-hot encoded as discussed
def tokenize_long_context(examples):
    # Tokenize and truncate to max length. 
    # Because it's 4096 tokens, very few legal documents will actually get truncated.
    return tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False, # We will pad dynamically in collate_fn
        add_special_tokens=True
    )

tokenized_datasets = hf_dataset.map(tokenize_long_context, batched=True, remove_columns=["text"])


In [ ]:
print(hf_dataset["train"][0])

In [ ]:

print(len(tokenized_datasets["train"][0]["input_ids"]))

In [ ]:
import torch
import numpy as np

print("Calculating class frequencies for pos_weight...")

# Total number of training cases
num_train_cases = len(hf_dataset["train"])
statute_counts = np.zeros(NUM_STATUTES)

# Count positive occurrences of each statute
for example in hf_dataset["train"]:
    if "pred" in example and example["pred"] is not None:
        for statute_id in example["pred"]:
            statute_counts[statute_id] += 1

# Formula for BCE pos_weight: (number of negative samples) / (number of positive samples)
# We add a small epsilon to prevent division by zero for statutes that might have 0 counts
epsilon = 1e-5
pos_weights = (num_train_cases - statute_counts) / (statute_counts + epsilon)

# PRO TIP: Clamp the maximum weight. 
# If a statute appears only 1 time in 30,000 cases, its weight would be 29,999. 
# This will cause explosive gradients (NaN loss). Capping it at ~50.0 is much safer.
pos_weights = np.clip(pos_weights, a_min=1.0, a_max=50.0)

# Convert to a PyTorch tensor
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)
print("pos_weight tensor created successfully!")

## 3. Model Initialization and QLORA adaptations

In [ ]:
!pip install causal-conv1d>=1.2.0
!pip install mamba-ssm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# --- 1. Import the official CUDA-optimized Mamba2 ---
from mamba_ssm import Mamba2

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 64, expand: int = 2, headdim: int = 64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        
        # --- 2. Use the official hardware-optimized SSM Mixer ---
        self.ssm = Mamba2(
            d_model=d_model,
            d_state=d_state,
            d_conv=4,
            expand=expand,
            headdim=headdim,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ssm(self.norm(x))

class Model3_SSM(nn.Module):
    """Hardware-optimized State-Space Model wrapper."""
    # Note: **kwargs absorbs leftover config arguments like 'scan_chunk' or 'variant' safely
    def __init__(self, vocab_size: int = 32000, d_model: int = 768, n_layers: int = 12,
                 d_state: int = 64, expand: int = 2, headdim: int = 64, grad_ckpt: bool = False, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.grad_ckpt = grad_ckpt

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, expand=expand, headdim=headdim)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None, last_only: bool = False):
        x = self.embedding(input_ids)
        for layer in self.layers:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.norm_f(x)
        
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1), ignore_index=0)
        return logits, loss

In [ ]:
import torch
import torch.nn as nn
import math

class LegalSSMForMultiLabel(nn.Module):
    def __init__(self, ssm_config, num_labels=NUM_STATUTES, pos_weight_tensor=None):
        super().__init__()
        
        self.ssm = Model3_SSM(**ssm_config)
        
        # Sever the tied weights before deleting lm_head (The Fix for Multi-GPU)
        clean_weight = self.ssm.embedding.weight.clone().detach()
        self.ssm.embedding.weight = nn.Parameter(clean_weight)
        del self.ssm.lm_head
        
        self.classifier = nn.Linear(self.ssm.d_model, num_labels)
        
        # --- THE FIX: Register the pos_weight tensor ---
        if pos_weight_tensor is not None:
            # register_buffer ensures it moves to the correct device automatically
            self.register_buffer('pos_weight', pos_weight_tensor)
        else:
            self.pos_weight = None

    def forward(self, input_ids, attention_mask, labels=None):
        x = self.ssm.embedding(input_ids)
        
        for layer in self.ssm.layers:
            if self.ssm.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
                
        x = self.ssm.norm_f(x)
        
        # Mean Pooling
        mask_expanded = attention_mask.unsqueeze(-1).float()
        x_masked = x * mask_expanded
        sum_embeddings = torch.sum(x_masked, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        
        doc_representation = sum_embeddings / sum_mask 
        logits = self.classifier(doc_representation)
        
        loss = None
        if labels is not None:
            # --- Apply pos_weight during loss calculation ---
            loss = F.binary_cross_entropy_with_logits(
                logits, 
                labels, 
                pos_weight=self.pos_weight
            )
            
        return {"logits": logits, "loss": loss}
        
# --- 3. Instantiation & Weight Loading ---

print("Initializing architecture...")
ssm_config = {
    "vocab_size":32000,
    "d_model":768,
    "n_layers":12,
    "d_state":64,
    "expand":2,
    "scan_chunk":64,
    "grad_ckpt":True,
    "variant":"mamba2",
    "headdim":64
}

model = LegalSSMForMultiLabel(ssm_config, num_labels=NUM_STATUTES,pos_weight_tensor=pos_weight_tensor)# Inject it here

# Load your Phase 1 pretrained SSM weights!
# Note: strict=False because we deleted lm_head and added classifier
pretrained_state = torch.load("/kaggle/input/datasets/vaibhavim081203/lastbreath245/model3_ssm_output/checkpoints/best_model_ssm.pt", map_location="cpu")
model.ssm.load_state_dict(pretrained_state, strict=False)

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")

## 4. Training loop and Evaluation

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW  # <-- Import from PyTorch instead
from tqdm.auto import tqdm
import random
import numpy as np
import os
import gc

# --- 1. DataLoader & Padding Setup ---

PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def collate_fn_long_context(batch):
    """Dynamically pads a 2D batch of sequences."""
    max_len = max(len(item['input_ids']) for item in batch)
    
    input_ids, attention_mask, labels, ids = [], [], [], []
    
    for item in batch:
        pad_len = max_len - len(item['input_ids'])
        
        input_ids.append(item['input_ids'] + [PAD_TOKEN_ID] * pad_len)
        # 1 for real tokens, 0 for pad tokens
        attention_mask.append(item['attention_mask'] + [0] * pad_len if 'attention_mask' in item else [1]*len(item['input_ids']) + [0]*pad_len)
        # --- THE FIX ---
        # Check that the key exists AND that it is not None
        if 'labels' in item and item['labels'] is not None:
            labels.append(item['labels'])
        if 'id' in item:
            ids.append(item['id'])
            
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.float32) if labels else None,
        'ids': ids
    }



# Create DataLoaders (adjust batch_size based on your GPU VRAM, e.g., 4, 8, or 16)
BATCH_SIZE = 1
train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collate_fn_long_context)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=BATCH_SIZE, collate_fn=collate_fn_long_context)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=BATCH_SIZE, collate_fn=collate_fn_long_context)


# --- 2. Training Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

TOTAL_INTENDED_EPOCHS = 3
EPOCHS_THIS_SESSION = 1 # Change this depending on Kaggle time limits
ACCUMULATION_STEPS = 16 # Effective batch size = 2 * 8 = 16

# Lower LR for full fine-tuning to prevent catastrophic forgetting
LEARNING_RATE = 3e-5 
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = (len(train_loader) // ACCUMULATION_STEPS) * TOTAL_INTENDED_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

# Mixed Precision Scaler
scaler = torch.amp.GradScaler('cuda')

start_epoch = 0
best_macro_f1 = 0.0
best_threshold= 0.0
# --- BULLETPROOF RESUME LOGIC ---
checkpoint_path = "/kaggle/input/cs781p5s2/fulltune_checkpoint.pt"

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    # weights_only=False required in PyTorch 2.6 to load numpy/random states
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if hasattr(model, 'module'):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_macro_f1 = checkpoint['best_macro_f1']
    
    # Safe RNG Restoration (.cpu().byte() prevents the TypeError)
    torch.set_rng_state(checkpoint['torch_rng_state'].cpu().byte())
    cuda_states = [state.cpu().byte() for state in checkpoint['cuda_rng_state']]
    torch.cuda.set_rng_state_all(cuda_states)
    np.random.set_state(checkpoint['numpy_rng_state'])
    random.setstate(checkpoint['python_rng_state'])
    print(f"Resumed successfully from Epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")
    
# --- 3. Training & Evaluation Loop ---

def evaluate_dynamic_threshold(model, dataloader):
    model.eval()
    all_preds_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids, attention_mask)
            
            probs = torch.sigmoid(outputs['logits'])
            all_preds_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    all_preds_probs = np.array(all_preds_probs)
    all_labels = np.array(all_labels)
    
    # Grid search for the best threshold
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        binary_preds = (all_preds_probs > thresh).astype(int)
        current_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = thresh
            
    return best_f1, best_thresh

# --- TRAINING LOOP ---
for epoch in range(start_epoch, start_epoch + EPOCHS_THIS_SESSION):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TOTAL_INTENDED_EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

       # AMP Forward pass
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids, attention_mask, labels=labels)
            
            loss = outputs['loss']
            
            # --- THE FIX ---
            # If using DataParallel, loss is a vector (one per GPU). Take the mean.
            if loss.dim() > 0:
                loss = loss.mean()
                
            # Now safely divide by accumulation steps
            loss = loss / ACCUMULATION_STEPS
    
        # AMP Backward pass
        scaler.scale(loss).backward()
        
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            # Unscale before clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Step scaler and optimizer
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()   
        # Logging
        actual_loss = loss.item() * ACCUMULATION_STEPS
        total_loss += actual_loss
        progress_bar.set_postfix({'loss': f"{actual_loss:.4f}"})
  
    avg_train_loss = total_loss / len(train_loader)
    
    # Evaluation
    val_macro_f1, best_threshold = evaluate_dynamic_threshold(model, val_loader)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} | Dev Macro-F1: {val_macro_f1:.4f} (Thresh: {best_threshold:.2f})")
    
    # Save Bulletproof Checkpoint
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    torch.save({
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_macro_f1': best_macro_f1,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all(),
        'numpy_rng_state': np.random.get_state(),
        'python_rng_state': random.getstate()
    }, "fulltune_checkpoint.pt")
    
    # Save "Best Model" specifically for Codabench submission
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        torch.save(model_state, "best_ssm_model.pt")
        print("--> New best model saved!")

## 5. Prediction for Test cases and submission

In [ ]:
import json
import zipfile
import torch
from tqdm.auto import tqdm

print("Generating predictions for the test set...")

# 1. Load best model weights safely
best_weights = torch.load("best_ssm_model.pt", map_location=device, weights_only=True)
model.load_state_dict(best_weights)
model.eval()

# 2. Load the optimal threshold we saved during training
# If you didn't save it to a file, just hardcode the best threshold you saw in the logs (e.g., best_t = 0.85)

best_t = best_threshold
print(f"Loaded optimal threshold: {best_t}")


predictions = []

# 3. Run inference
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)      
        
        # Use AMP for inference speed and safety
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            
            # Apply Sigmoid for MULTI-LABEL probabilities
            probs = torch.sigmoid(logits)
        
        # Apply the threshold to create a binary multi-hot vector (0s and 1s)
        binary_preds = (probs > best_t).int().cpu().tolist()
        
        # Extract the case IDs 
        batch_ids = batch['ids']
        
        # Now 'pred' is a list of 0s and 1s for all NUM_STATUTES
        for case_id, pred in zip(batch_ids, binary_preds):
            predictions.append({
                "id": case_id,
                "pred": pred 
            })

# 4. Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(predictions, f, indent=4)

print(f"Predictions saved to {output_file}.")

# 5. Zip the file for Codabench submission
zip_filename = "submission.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # arcname ensures the file sits at the root of the zip
    zipf.write(output_file, arcname=output_file)

print(f"Successfully created {zip_filename}! You are ready to submit to Codabench.")